In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import (
    chi2_contingency,
    fisher_exact,
    spearmanr,
)

from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
)

In [ ]:
GROUND_TRUTH_DIR = Path(".")

'''
"posts_set": pd.read_csv(
    GROUND_TRUTH_DIR / "POSTS_subset_values.csv"
)
'''
ground_truth_dataframes = {
    "essay_fullset": pd.read_csv(
        GROUND_TRUTH_DIR / "ESSAYS_fullset_values.csv"
    ),
    "essay_subset": pd.read_csv(
        GROUND_TRUTH_DIR / "ESSAYS_subset_values.csv"
    ),
    "facebook_subset": pd.read_csv(
        GROUND_TRUTH_DIR / "FACEBOOK_subset_values.csv"
    ),
}

In [ ]:
OUTPUT_DIR = Path("personality_outputs")

prompt_configs = [
    ("original_basic"),
    ("original_basic_ex"),
    ("ling_basic"),
    ("ling_basic_ex"),
]

model_configs = [
    ("gpt4o_mini"),
    ("o3_mini"),
    ("gpt54_mini"),
    ("gpt4o"),
    ("claude_haiku"),
]

'''
(
    "posts_text",
    posts_text.iloc[:, -1].astype(str),
),
'''
'''
(
    "ablated_posts",
    ablated_posts.iloc[:, -1].astype(str),
)
'''
corpus_configs = [
    (
        "essay_fullset"
    ),
    (
        "essay_subset"
    ),
    (
        "facebook_subset"
    ),
    (
        "ablated_essay_fullset"
    ),
    (
        "ablated_essay_subset"
    ),
    (
        "ablated_facebook_subset"
    ),
]


generated_filenames = [
    f"{prompt_name}_{model_name}_{corpus_name}.csv"
    for corpus_name in corpus_configs
    for prompt_name in prompt_configs
    for model_name in model_configs
]

generated_dataframes = {
    Path(filename).stem: pd.read_csv(
        OUTPUT_DIR / filename
    )
    for filename in generated_filenames
}

print(generated_filenames)

In [ ]:
generated_dataframes[
    "original_basic_claude_haiku_essay_fullset"
]

In [ ]:
EVALUATION_DIR = Path("evaluation_results")
EVALUATION_DIR.mkdir(exist_ok=True)

TRAIT_COLUMN_MAP = {
    "openness": "Openness_classification",
    "conscientiousness": "Conscientiousness_classification",
    "extroversion": "Extroversion_classification",
    "agreeableness": "Agreeableness_classification",
    "neuroticism": "Neuroticism_classification",
}

In [ ]:
# Performs the statistical analyses outlined in the Performance and Association Metrics section, 
# leading to the results presented in the Prediction and Ablation Results section.

all_metrics = []

corpus_ground_truth_map = {
    "essay_fullset": "essay_fullset",
    "essay_subset": "essay_subset",
    "facebook_subset": "facebook_subset"
}

for config_name, predictions_df in generated_dataframes.items():
    matching_corpora = [
        corpus_name
        for corpus_name in corpus_ground_truth_map
        if config_name.endswith(f"_{corpus_name}")
    ]
    
    corpus_name = max(matching_corpora, key=len)
    
    ground_truth_name = corpus_ground_truth_map[corpus_name]
    ground_truth_df = ground_truth_dataframes[ground_truth_name]

    corpus_name = matching_corpora[0]

    predictions = predictions_df.copy()
    ground_truth = ground_truth_dataframes[
        corpus_name
    ].copy()

    if "source_index" not in ground_truth.columns:
        ground_truth = ground_truth.reset_index(
            names="source_index"
        )

    required_prediction_columns = {
        "source_index",
        "status",
        *TRAIT_COLUMN_MAP.values(),
    }

    missing_prediction_columns = (
        required_prediction_columns
        - set(predictions.columns)
    )

    if missing_prediction_columns:
        raise ValueError(
            f"{config_name} is missing prediction columns: "
            f"{sorted(missing_prediction_columns)}"
        )

    required_ground_truth_columns = {
        "source_index",
        *TRAIT_COLUMN_MAP.keys(),
    }

    missing_ground_truth_columns = (
        required_ground_truth_columns
        - set(ground_truth.columns)
    )

    if missing_ground_truth_columns:
        raise ValueError(
            f"{corpus_name} is missing ground-truth columns: "
            f"{sorted(missing_ground_truth_columns)}"
        )

    if predictions["source_index"].duplicated().any():
        raise ValueError(
            f"{config_name} has duplicate source_index values"
        )

    if ground_truth["source_index"].duplicated().any():
        raise ValueError(
            f"{corpus_name} has duplicate source_index values"
        )

    successful_predictions = predictions[
        predictions["status"] == "ok"
    ].copy()

    failed_predictions = predictions[
        predictions["status"] != "ok"
    ].copy()

    comparison = ground_truth[
        [
            "source_index",
            *TRAIT_COLUMN_MAP.keys(),
        ]
    ].merge(
        successful_predictions[
            [
                "source_index",
                *TRAIT_COLUMN_MAP.values(),
            ]
        ],
        on="source_index",
        how="inner",
        validate="one_to_one",
    )

    if comparison.empty:
        raise ValueError(
            f"{config_name} has no successful predictions "
            f"matched to ground truth"
        )

    n_ground_truth = len(ground_truth)
    n_output_rows = len(predictions)
    n_successful = len(successful_predictions)
    n_failed = len(failed_predictions)
    n_matched = len(comparison)
    n_missing_predictions = n_ground_truth - n_matched
    n_unmatched_successes = n_successful - n_matched

    configuration_metrics = []

    for truth_column, prediction_column in (
        TRAIT_COLUMN_MAP.items()
    ):
        y_true = pd.to_numeric(
            comparison[truth_column],
            errors="coerce",
        )

        y_pred = pd.to_numeric(
            comparison[prediction_column],
            errors="coerce",
        )

        if y_true.isna().any():
            raise ValueError(
                f"{corpus_name} has missing/non-numeric "
                f"ground truth for {truth_column}"
            )

        if y_pred.isna().any():
            raise ValueError(
                f"{config_name} has missing/non-numeric "
                f"predictions for {prediction_column}"
            )

        y_true = y_true.astype(int)
        y_pred = y_pred.astype(int)

        true_values = set(y_true.unique())
        predicted_values = set(y_pred.unique())

        if not true_values.issubset({0, 1}):
            raise ValueError(
                f"{corpus_name}/{truth_column} has "
                f"nonbinary values: {true_values}"
            )

        if not predicted_values.issubset({0, 1}):
            raise ValueError(
                f"{config_name}/{prediction_column} has "
                f"nonbinary values: {predicted_values}"
            )

        accuracy = accuracy_score(y_true, y_pred)

        f1 = f1_score(
            y_true,
            y_pred,
            zero_division=0,
        )

        tn, fp, fn, tp = confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1],
        ).ravel()

        if (
            y_true.nunique() == 2
            and y_pred.nunique() == 2
        ):
            spearman_r, spearman_p = spearmanr(
                y_true,
                y_pred,
            )
        else:
            spearman_r = np.nan
            spearman_p = np.nan

        if (
            y_true.nunique() == 2
            and y_pred.nunique() == 2
        ):
            contingency = pd.crosstab(
                y_true,
                y_pred,
            ).reindex(
                index=[0, 1],
                columns=[0, 1],
                fill_value=0,
            )

            fisher_odds_ratio, fisher_p = (
                fisher_exact(contingency)
            )

            chi2, chi2_p, chi2_dof, _ = (
                chi2_contingency(contingency)
            )
        else:
            fisher_odds_ratio = np.nan
            fisher_p = np.nan
            chi2 = np.nan
            chi2_p = np.nan
            chi2_dof = np.nan

        if (
            y_true.nunique() == 1
            and y_pred.nunique() == 1
        ):
            kappa = np.nan
        else:
            kappa = cohen_kappa_score(
                y_true,
                y_pred,
            )

        configuration_metrics.append({
            "configuration": config_name,
            "corpus": corpus_name,
            "trait": truth_column,
            "n_ground_truth": n_ground_truth,
            "n_output_rows": n_output_rows,
            "n_successful": n_successful,
            "n_failed": n_failed,
            "n_matched": n_matched,
            "n_missing_predictions": n_missing_predictions,
            "n_unmatched_successes": n_unmatched_successes,
            "accuracy": accuracy,
            "f1": f1,
            "spearman_r": spearman_r,
            "spearman_p": spearman_p,
            "fisher_odds_ratio": fisher_odds_ratio,
            "fisher_p": fisher_p,
            "cohen_kappa": kappa,
            "chi2": chi2,
            "chi2_p": chi2_p,
            "chi2_dof": chi2_dof,
            "true_negative": tn,
            "false_positive": fp,
            "false_negative": fn,
            "true_positive": tp,
        })

    metrics_df = pd.DataFrame(
        configuration_metrics
    )

    metrics_df.to_csv(
        EVALUATION_DIR
        / f"{config_name}_metrics.csv",
        index=False,
    )

    all_metrics.append(metrics_df)

all_metrics_df = pd.concat(
    all_metrics,
    ignore_index=True,
)

all_metrics_df.to_csv(
    EVALUATION_DIR / "all_configuration_metrics.csv",
    index=False,
)

print(
    f"Saved {len(all_metrics)} configuration files "
    f"to {EVALUATION_DIR.resolve()}"
)